# Food Plant Experiment & Causal Analysis

## Introduction

This project focuses on analysing manufacturing performance using A/B testing and statistical methods. The main objective is to measure whether a new manufacturing initiative improves production efficiency, reduces wastage and maintains product quality.

Python is used for data cleaning, statistical analysis and visualisation. The project uses a simulated manufacturing dataset containing approximately 300,000 production records from three production lines.

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy statsmodels

## Step 2: Import Required Libraries

Python libraries are used to perform data manipulation, numerical calculations, statistical testing and data visualisation.

In [ ]:
#Import important labraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.power import TTestIndPower

print("All libraries imported successfully!")

## Step 3: Dataset Generation

A simulated manufacturing dataset is created using Python. It contains 300,000 records from three production lines, including supplier details, raw material usage, production yield, wastage, quality rejection and machine downtime.

In [ ]:
#Import dataset
import pandas as pd
import numpy as np

np.random.seed(42)

n = 300000

# Generate basic production information
dates = pd.date_range("2026-01-01", "2026-09-30")
production_dates = np.random.choice(dates, n)

df = pd.DataFrame({
    "batch_id": range(1, n + 1),
    "date": production_dates,
    "production_line": np.random.choice(
        ["Line A", "Line B", "Line C"], n
    ),
    "region": np.random.choice(
        ["Jaipur", "Ajmer"], n
    ),
    "supplier_name": np.random.choice(
        ["ABC Foods", "abc foods", "ABC FOOD",
         "Fresh Suppliers", "fresh suppliers",
         "Quality Raw Materials"], n
    ),
    "shift": np.random.choice(
        ["Morning", "Evening", "Night"], n
    ),
    "machine": np.random.choice(
        ["Machine 1", "Machine 2", "Machine 3"], n
    )
})

# Historical business initiative
df["post"] = (df["date"] >= "2026-07-01").astype(int)

df["region_treatment"] = (
    df["region"] == "Jaipur"
).astype(int)

df["did_group"] = (
    df["region_treatment"] * df["post"]
)

# Randomised A/B experiment in September
df["ab_group"] = "Not in Experiment"

experiment = df["date"] >= "2026-09-01"

df.loc[experiment, "ab_group"] = np.random.choice(
    ["Control", "Treatment"],
    experiment.sum()
)

# Production quantities
df["raw_material_kg"] = np.random.normal(
    1000, 100, n
).clip(500, 1500)

# Simulate seasonal and production effects
line_effect = df["production_line"].map({
    "Line A": 0.02,
    "Line B": 0.00,
    "Line C": -0.02
})

season_effect = np.where(
    df["date"].dt.month.isin([6, 7, 8]),
    -0.01, 0
)

historical_effect = (
    0.015 * df["did_group"]
)

ab_effect = np.where(
    df["ab_group"] == "Treatment",
    0.01, 0
)

base_yield = (
    0.80 + line_effect + season_effect
    + historical_effect + ab_effect
)

df["process_yield"] = np.clip(
    base_yield + np.random.normal(0, 0.04, n),
    0.60, 0.98
)

df["good_output_kg"] = (
    df["raw_material_kg"] * df["process_yield"]
)

df["waste_kg"] = (
    df["raw_material_kg"] - df["good_output_kg"]
)

df["wastage_percentage"] = (
    df["waste_kg"] / df["raw_material_kg"]
) * 100

# Quality rejection
df["quality_rejected_kg"] = (
    df["good_output_kg"] *
    np.random.uniform(0.005, 0.04, n)
)

df["quality_rejection_rate"] = (
    df["quality_rejected_kg"] /
    df["good_output_kg"]
) * 100

# Moisture readings
df["moisture_percentage"] = np.random.normal(
    12, 2, n
)

# Missing moisture readings during night shifts
night_mask = df["shift"] == "Night"

missing_mask = (
    night_mask &
    (np.random.random(n) < 0.25)
)

df.loc[missing_mask, "moisture_percentage"] = np.nan

# Machine downtime
df["machine_downtime_min"] = np.random.exponential(
    15, n
).clip(0, 120)

# Mixed weight units
df["raw_material_value"] = np.where(
    np.random.random(n) < 0.5,
    df["raw_material_kg"],
    df["raw_material_kg"] * 1000
)

df["weight_unit"] = np.where(
    df["raw_material_value"] > 1500,
    "grams", "kg"
)

# Save dataset
df.to_csv("manufacturing_data.csv", index=False)

print("Dataset generated successfully!")
print("Total records:", len(df))
print("Total columns:", len(df.columns))

print(df.head())


In [ ]:
#dataset information
print("Dataset Information:")
print(df.info())

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nProduction Lines:")
print(df["production_line"].value_counts())

## Step 4: Data Cleaning and Preprocessing

Data cleaning is performed to improve dataset quality. Duplicate records are removed, supplier names are standardised, weight units are converted into kilograms, and missing moisture readings are handled. Invalid values are also checked before further analysis.

In [ ]:
#Step 4: Data Cleaning and Preprocessing

print("Original Dataset Shape:", df.shape)

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nSupplier Names:")
print(df["supplier_name"].value_counts())

print("\nWeight Units:")
print(df["weight_unit"].value_counts())


In [ ]:
#Remove duplicate records

df = df.drop_duplicates(subset="batch_id")

print("Duplicate records removed.")
print("Remaining records:", len(df))


In [ ]:
#Standardise supplier names

df["supplier_name"] = (
    df["supplier_name"]
    .str.strip()
    .str.lower()
)

supplier_mapping = {
    "abc food": "ABC Foods",
    "abc foods": "ABC Foods",
    "fresh suppliers": "Fresh Suppliers",
    "quality raw materials": "Quality Raw Materials"
}

df["supplier_name"] = df["supplier_name"].replace(
    supplier_mapping
)

print(df["supplier_name"].value_counts())


In [ ]:
#Convert all weights into kilograms

df["raw_material_kg_clean"] = np.where(
    df["weight_unit"] == "grams",
    df["raw_material_value"] / 1000,
    df["raw_material_value"]
)

print("Weight conversion completed.")

print(df[
    ["raw_material_value",
     "weight_unit",
     "raw_material_kg_clean"]
].head(10))


In [ ]:
#Handle missing moisture readings

print("Missing values before cleaning:")
print(df["moisture_percentage"].isnull().sum())

df["moisture_percentage"] = (
    df.groupby("shift")["moisture_percentage"]
    .transform(lambda x: x.fillna(x.median()))
)

print("\nMissing values after cleaning:")
print(df["moisture_percentage"].isnull().sum())


In [ ]:
#Check invalid values

print("Invalid Process Yield:")
print((df["process_yield"] < 0).sum())

print("Invalid Wastage Percentage:")
print((df["wastage_percentage"] < 0).sum())

print("Invalid Moisture Values:")
print((df["moisture_percentage"] < 0).sum())

print("Invalid Quality Rejection Rate:")
print((df["quality_rejection_rate"] < 0).sum())


In [ ]:
#Create the final clean dataset

df.to_csv("cleaned_manufacturing_data.csv", index=False)

print("Data cleaning completed successfully!")

print("Final Dataset Shape:", df.shape)

print("\nFinal Missing Values:")
print(df.isnull().sum().sum())



In [ ]:
from google.colab import files

files.download("cleaned_manufacturing_data.csv")

# Step 5: A/B Test Design and Power Analysis

A/B testing is used to compare two groups and measure the impact of a new manufacturing initiative.

- Control Group: Follows the existing production process.
- Treatment Group: Uses the new manufacturing initiative.

**Hypothesis:**
- H0: The new initiative does not improve process yield.
- H1: The new initiative improves process yield.

**Primary Metric:** Process Yield (%).

**Secondary Metrics:** Wastage Percentage and Quality Rejection Rate.

**Guardrail Metrics:** Machine Downtime and Moisture Percentage.

**Power Analysis:** Sample size is calculated using a 5% significance level, 80% statistical power and a 1 percentage-point minimum detectable effect.

**Experiment Duration:** The required testing duration is estimated using the sample size and average daily production volume. A minimum of 14 days is considered to cover weekly production patterns.

Finally, the experiment results will be evaluated using statistical tests and confidence intervals.

In [ ]:

# STEP 5: SAMPLE SIZE AND POWER ANALYSIS

# Use pre-experiment data to estimate baseline variation
baseline = df[df["date"] < "2026-09-01"]

baseline_std = baseline["process_yield"].std()

# Experiment assumptions
alpha = 0.05
power = 0.80
mde = 0.01

# Calculate Cohen's effect size
effect_size = mde / baseline_std

# Power analysis
analysis = TTestIndPower()

sample_size = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1,
    alternative="larger"
)

sample_per_group = int(np.ceil(sample_size))

total_sample = sample_per_group * 2

print("SAMPLE SIZE CALCULATION")
print("-----------------------")
print("Baseline standard deviation:", round(baseline_std, 4))
print("Minimum detectable effect:", mde)
print("Cohen's effect size:", round(effect_size, 4))
print("Significance level:", alpha)
print("Statistical power:", power)
print("Required Control samples:", sample_per_group)
print("Required Treatment samples:", sample_per_group)
print("Total required samples:", total_sample)


In [ ]:

# Calculate daily production volume

total_days = (
    df["date"].max() - df["date"].min()
).days + 1

daily_batches = len(df) / total_days

# Calculate required experiment duration
required_days = np.ceil(
    total_sample / daily_batches
)

# Minimum duration to cover weekly production patterns
recommended_days = max(14, int(required_days))

# Available experiment period
experiment_data = df[
    df["date"] >= "2026-09-01"
]

available_days = experiment_data["date"].nunique()

print("EXPERIMENT DURATION")
print("-------------------")
print("Average daily batches:", round(daily_batches))
print("Required sample:", total_sample)
print("Calculated duration:", int(required_days), "days")
print("Recommended duration:", recommended_days, "days")
print("Available experiment duration:", available_days, "days")


In [ ]:

# Select experiment records
experiment_data = df[
    df["date"] >= "2026-09-01"
].copy()

# Count samples in each group
group_counts = experiment_data["ab_group"].value_counts()

print("A/B TEST GROUP DISTRIBUTION")
print(group_counts)

control_count = group_counts.get("Control", 0)
treatment_count = group_counts.get("Treatment", 0)

print("\nRequired samples per group:", sample_per_group)

print("Control samples available:", control_count)
print("Treatment samples available:", treatment_count)

if control_count >= sample_per_group and treatment_count >= sample_per_group:
    print("\nResult: Required sample size is available.")
else:
    print("\nResult: More experiment data is required.")


In [ ]:

design_results = pd.DataFrame({
    "Parameter": [
        "Experiment Type",
        "Primary Metric",
        "Significance Level",
        "Statistical Power",
        "Minimum Detectable Effect",
        "Baseline Standard Deviation",
        "Required Control Samples",
        "Required Treatment Samples",
        "Total Required Samples",
        "Recommended Duration"
    ],
    "Value": [
        "Randomised A/B Test",
        "Process Yield",
        alpha,
        power,
        mde,
        baseline_std,
        sample_per_group,
        sample_per_group,
        total_sample,
        recommended_days
    ]
})

design_results.to_csv(
    "experiment_design_results.csv",
    index=False
)

print(design_results)

print("\nExperiment design saved successfully!")


In [ ]:
files.download("experiment_design_results.csv")

## Step 6: Experiment Data Analysis

In this step, the experimental data is analysed to compare the Control and Treatment groups. Randomisation balance and Sample Ratio Mismatch (SRM) are checked to identify any distribution problems. Statistical tests, including independent t-tests, are used to measure the treatment effect. A 95% confidence interval is calculated to understand the reliability of the results.

In [ ]:

experiment_data = df[
    df["date"] >= "2026-09-01"
].copy()

print("Experiment Dataset:", experiment_data.shape)

print("\nGroup Distribution:")
print(experiment_data["ab_group"].value_counts())


In [ ]:
#Check randomisation balance
balance_columns = [
    "production_line",
    "supplier_name",
    "shift"
]

for column in balance_columns:
    print("\nBalance Check:", column)

    print(pd.crosstab(
        experiment_data["ab_group"],
        experiment_data[column],
        normalize="index"
    ).round(3))


In [ ]:
#Sample Ratio Mismatch (SRM)

from scipy.stats import chisquare

counts = experiment_data[
    "ab_group"
].value_counts().reindex(
    ["Control", "Treatment"],
    fill_value=0
)

chi_stat, p_value = chisquare(
    counts.values,
    f_exp=[counts.sum()/2, counts.sum()/2]
)

print("Control:", counts["Control"])
print("Treatment:", counts["Treatment"])
print("Chi-square:", chi_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("SRM detected: Investigate group allocation.")
else:
    print("No statistically significant SRM detected.")


In [ ]:
#Compare process yield
group_summary = experiment_data.groupby(
    "ab_group"
)["process_yield"].agg(
    ["count", "mean", "std"]
)

print(group_summary)

control_mean = experiment_data[
    experiment_data["ab_group"] == "Control"
]["process_yield"].mean()

treatment_mean = experiment_data[
    experiment_data["ab_group"] == "Treatment"
]["process_yield"].mean()

print("\nControl Mean Yield:", control_mean)
print("Treatment Mean Yield:", treatment_mean)

print(
    "Treatment Effect:",
    treatment_mean - control_mean
)


In [ ]:
#Perform independent t-test

control = experiment_data.loc[
    experiment_data["ab_group"] == "Control",
    "process_yield"
]

treatment = experiment_data.loc[
    experiment_data["ab_group"] == "Treatment",
    "process_yield"
]

t_stat, p_value = stats.ttest_ind(
    treatment,
    control,
    equal_var=False,
    alternative="greater"
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Statistically significant improvement detected.")
else:
    print("No statistically significant improvement detected.")


In [ ]:
#Calculate 95% confidence interval
effect = treatment.mean() - control.mean()

se = np.sqrt(
    treatment.var(ddof=1) / len(treatment) +
    control.var(ddof=1) / len(control)
)

df_welch = (
    (treatment.var(ddof=1)/len(treatment) +
     control.var(ddof=1)/len(control))**2
) / (
    (treatment.var(ddof=1)/len(treatment))**2 /
    (len(treatment)-1) +
    (control.var(ddof=1)/len(control))**2 /
    (len(control)-1)
)

critical_value = stats.t.ppf(
    0.975, df_welch
)

lower = effect - critical_value * se
upper = effect + critical_value * se

print("Treatment Effect:", effect)
print("95% Confidence Interval:", (lower, upper))


In [ ]:
#Visualise the results
sns.barplot(
    data=experiment_data,
    x="ab_group",
    y="process_yield",
    errorbar=("ci", 95)
)

plt.title("Control vs Treatment Process Yield")
plt.xlabel("Experiment Group")
plt.ylabel("Process Yield")
plt.show()


In [ ]:
#Save experiment results
step6_results = pd.DataFrame({
    "Metric": [
        "Control Mean Yield",
        "Treatment Mean Yield",
        "Treatment Effect",
        "T Statistic",
        "P Value",
        "95% CI Lower",
        "95% CI Upper",
        "SRM P Value"
    ],
    "Value": [
        control_mean,
        treatment_mean,
        effect,
        t_stat,
        p_value,
        lower,
        upper,
        globals()["p_value"] if False else chi_stat
    ]
})

step6_results.to_csv(
    "experiment_analysis_results.csv",
    index=False
)

print("Step 6 results saved successfully!")


In [ ]:
srm_p_value = p_value

## Step 7: Difference-in-Differences Analysis

Difference-in-Differences (DiD) is used to estimate the impact of a past business change by comparing two regions over time.

- Treatment Region: Jaipur
- Control Region: Ajmer
- Pre-change Period: January–June 2026
- Post-change Period: July–August 2026

The parallel-trends assumption checks whether both regions followed similar trends before the change. The DiD estimate measures the difference in their changes in process yield after the initiative.

In [ ]:
#Prepare the DiD dataset

import statsmodels.formula.api as smf

did_df = df[df["date"] < "2026-09-01"].copy()

did_df["region_treatment"] = (
    did_df["region"] == "Jaipur"
).astype(int)

did_df["post"] = (
    did_df["date"] >= "2026-07-01"
).astype(int)

did_df["did_group"] = (
    did_df["region_treatment"] * did_df["post"]
)

print("DiD dataset shape:", did_df.shape)
print("\nRegion distribution:")
print(did_df["region"].value_counts())

print("\nPeriod distribution:")
print(did_df["post"].value_counts())

In [ ]:
#Check parallel trends visually
pre_data = did_df[did_df["post"] == 0].copy()

pre_data["month"] = (
    pre_data["date"].dt.to_period("M").dt.to_timestamp()
)

monthly_pre = (
    pre_data.groupby(["month", "region"])["process_yield"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 5))

sns.lineplot(
    data=monthly_pre,
    x="month",
    y="process_yield",
    hue="region",
    marker="o"
)

plt.title("Parallel Trends Check: Jaipur vs Ajmer")
plt.xlabel("Month")
plt.ylabel("Average Process Yield")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
#Test the parallel-trends assumption
pre_data["month_index"] = pre_data["date"].dt.month
pre_data["jaipur"] = (
    pre_data["region"] == "Jaipur"
).astype(int)

pretrend_model = smf.ols(
    "process_yield ~ jaipur * month_index",
    data=pre_data
).fit(cov_type="HC3")

print(pretrend_model.summary())

pretrend_p = pretrend_model.pvalues["jaipur:month_index"]

print("\nParallel Trends Test P-value:", pretrend_p)

if pretrend_p >= 0.05:
    print("No significant difference in pre-change trends was detected.")
else:
    print("A significant difference in pre-change trends was detected.")

In [ ]:
#calculate the DiD effect
did_model = smf.ols(
    "process_yield ~ region_treatment + post + did_group",
    data=did_df
).fit(cov_type="HC3")

print(did_model.summary())

did_effect = did_model.params["did_group"]
did_p_value = did_model.pvalues["did_group"]

did_ci = did_model.conf_int().loc["did_group"]

print("\nDiD Results")
print("Estimated Treatment Effect:", did_effect)
print("Effect in Percentage Points:", did_effect * 100)
print("P-value:", did_p_value)
print("95% Confidence Interval:", did_ci.values)
print(
    "95% CI in Percentage Points:",
    did_ci.values * 100
)

In [ ]:
did_results = pd.DataFrame({
    "Metric": [
        "DiD Effect",
        "DiD Effect (Percentage Points)",
        "P-value",
        "95% CI Lower",
        "95% CI Upper",
        "Parallel Trends P-value"
    ],
    "Value": [
        did_effect,
        did_effect * 100,
        did_p_value,
        did_ci.iloc[0] * 100,
        did_ci.iloc[1] * 100,
        pretrend_p
    ]
})

did_results.to_csv("did_analysis_results.csv", index=False)

print("Step 7 results saved successfully!")

## Step 8: Segment Analysis

Segment analysis is performed to understand how the new manufacturing initiative affects different groups.

The treatment effect is compared across production lines, suppliers, shifts and regions. The analysis identifies which groups benefit from the initiative and which groups may experience lower performance.

For each segment, the average process yield, treatment effect and 95% confidence interval are calculated. This helps identify differences in manufacturing performance across groups.

In [ ]:

# STEP 8: SEGMENT ANALYSIS

# Select experiment data
segment_data = df[
    df["date"] >= "2026-09-01"
].copy()

# Define segments
segments = [
    "production_line",
    "supplier_name",
    "shift",
    "region"
]

segment_results = []

# Analyse each segment
for segment in segments:

    for group in segment_data[segment].dropna().unique():

        data = segment_data[
            segment_data[segment] == group
        ]

        control = data.loc[
            data["ab_group"] == "Control",
            "process_yield"
        ]

        treatment = data.loc[
            data["ab_group"] == "Treatment",
            "process_yield"
        ]

        if len(control) < 2 or len(treatment) < 2:
            continue

        # Calculate treatment effect
        effect = treatment.mean() - control.mean()

        # Standard error
        se = np.sqrt(
            treatment.var(ddof=1) / len(treatment) +
            control.var(ddof=1) / len(control)
        )

        # Welch degrees of freedom
        df_welch = (
            (treatment.var(ddof=1) / len(treatment) +
             control.var(ddof=1) / len(control)) ** 2
        ) / (
            (treatment.var(ddof=1) / len(treatment)) ** 2 /
            (len(treatment) - 1) +
            (control.var(ddof=1) / len(control)) ** 2 /
            (len(control) - 1)
        )

        # 95% Confidence Interval
        critical = stats.t.ppf(0.975, df_welch)

        lower = effect - critical * se
        upper = effect + critical * se

        segment_results.append({
            "Segment Type": segment,
            "Segment": group,
            "Control Samples": len(control),
            "Treatment Samples": len(treatment),
            "Control Yield": control.mean(),
            "Treatment Yield": treatment.mean(),
            "Treatment Effect": effect,
            "CI Lower": lower,
            "CI Upper": upper
        })

# Create results table
segment_results = pd.DataFrame(segment_results)

print("SEGMENT ANALYSIS RESULTS")
print(segment_results.round(4))

# Save results
segment_results.to_csv(
    "segment_analysis_results.csv",
    index=False
)

print("\nSegment analysis completed successfully!")


In [ ]:
from google.colab import files

files.download("segment_analysis_results.csv")

## Step 9: CUPED / Regression Adjustment

CUPED (Controlled-experiment Using Pre-Experiment Data) and regression adjustment are statistical techniques used to reduce variance in A/B testing.

In this step, regression adjustment is applied using available pre-treatment characteristics such as production line, supplier, shift, region and raw material quantity.

The adjusted treatment effect is compared with the original treatment effect. Variance reduction is measured to estimate how much the required sample size and experiment duration could be reduced.

The adjusted treatment effect is reported with a 95% confidence interval.

In [ ]:

# STEP 9: REGRESSION ADJUSTMENT

# Select experimental data
cuped_data = df[
    (df["date"] >= "2026-09-01") &
    (df["ab_group"].isin(["Control", "Treatment"]))
].copy()

# Calculate original treatment effect
control_mean = cuped_data.loc[
    cuped_data["ab_group"] == "Control",
    "process_yield"
].mean()

treatment_mean = cuped_data.loc[
    cuped_data["ab_group"] == "Treatment",
    "process_yield"
].mean()

original_effect = treatment_mean - control_mean

# Original unadjusted regression
original_model = smf.ols(
    "process_yield ~ C(ab_group, Treatment(reference='Control'))",
    data=cuped_data
).fit(cov_type="HC3")

treatment_term = (
    "C(ab_group, Treatment(reference='Control'))[T.Treatment]"
)

original_ci = original_model.conf_int().loc[treatment_term]

# Regression adjustment using pre-treatment characteristics
adjusted_model = smf.ols(
    """process_yield ~
    C(ab_group, Treatment(reference='Control'))
    + C(production_line)
    + C(supplier_name)
    + C(shift)
    + C(region)
    + C(machine)
    + raw_material_kg_clean""",
    data=cuped_data
).fit(cov_type="HC3")

adjusted_effect = adjusted_model.params[treatment_term]

adjusted_ci = adjusted_model.conf_int().loc[treatment_term]

adjusted_p = adjusted_model.pvalues[treatment_term]

# Estimate variance reduction using covariates
covariate_model = smf.ols(
    """process_yield ~
    C(production_line)
    + C(supplier_name)
    + C(shift)
    + C(region)
    + C(machine)
    + raw_material_kg_clean""",
    data=cuped_data
).fit()

variance_reduction = covariate_model.rsquared

variance_ratio = 1 - variance_reduction

# Estimate adjusted sample size
adjusted_sample = int(np.ceil(
    total_sample * variance_ratio
))

# Estimate adjusted experiment duration
raw_duration = recommended_days

adjusted_duration = max(
    14,
    int(np.ceil(raw_duration * variance_ratio))
)

# Display results
print("REGRESSION ADJUSTMENT RESULTS")
print("--------------------------------")

print("Original Treatment Effect:", original_effect)
print("Original 95% CI:", original_ci.values)

print("\nAdjusted Treatment Effect:", adjusted_effect)
print("Adjusted 95% CI:", adjusted_ci.values)
print("Adjusted P-value:", adjusted_p)

print("\nVariance Reduction:",
      round(variance_reduction * 100, 2), "%")

print("Original Required Sample:", total_sample)
print("Adjusted Estimated Sample:", adjusted_sample)

print("\nOriginal Test Duration:", raw_duration, "days")
print("Adjusted Estimated Duration:", adjusted_duration, "days")

print("Estimated Days Saved:",
      max(0, raw_duration - adjusted_duration))


In [ ]:

# Save Step 9 results

step9_results = pd.DataFrame({
    "Metric": [
        "Original Treatment Effect",
        "Original CI Lower",
        "Original CI Upper",
        "Adjusted Treatment Effect",
        "Adjusted CI Lower",
        "Adjusted CI Upper",
        "Adjusted P-value",
        "Variance Reduction Percentage",
        "Original Required Sample",
        "Adjusted Estimated Sample",
        "Original Test Duration",
        "Adjusted Test Duration",
        "Estimated Days Saved"
    ],
    "Value": [
        original_effect,
        original_ci.iloc[0],
        original_ci.iloc[1],
        adjusted_effect,
        adjusted_ci.iloc[0],
        adjusted_ci.iloc[1],
        adjusted_p,
        variance_reduction * 100,
        total_sample,
        adjusted_sample,
        raw_duration,
        adjusted_duration,
        max(0, raw_duration - adjusted_duration)
    ]
})

step9_results.to_csv(
    "regression_adjustment_results.csv",
    index=False
)

print(step9_results.round(4))

print("\nStep 9 results saved successfully!")


In [ ]:

# Compare original and adjusted effects

comparison = pd.DataFrame({
    "Method": ["Original", "Adjusted"],
    "Treatment Effect": [
        original_effect,
        adjusted_effect
    ]
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=comparison,
    x="Method",
    y="Treatment Effect"
)

plt.axhline(0, color="black", linestyle="--")

plt.title("Original vs Adjusted Treatment Effect")
plt.xlabel("Analysis Method")
plt.ylabel("Treatment Effect on Process Yield")

plt.show()


In [ ]:
from google.colab import files

files.download("regression_adjustment_results.csv")

Step 10: Final Decision Memo
Objective
To summarize the results of the food plant experiment and recommend whether the company should implement the production change.

1. Experiment Summary
The experiment compared the Control group and Treatment group to understand whether the production change improved process yield and reduced wastage. It ran for 30 days in September 2026, gathering 33,086 batches (Control: 16,803, Treatment: 16,283).

2. Key Findings
Randomization and sample balance: A Chi-Square test for Sample Ratio Mismatch (SRM) detected a statistically significant discrepancy ( p=0.004 ) in group allocation (Control: 50.8%, Treatment: 49.2%). This requires engineering investigation into the randomization pipeline before complete rollout.
Treatment effect on process yield: The average process yield for the Control group was 80.74%, and the Treatment group was 81.71%. This represents a statistically significant, positive treatment effect of +0.97 percentage points ( t -stat:  19.96 ,  p≈0.0 ), with a 95% Confidence Interval of [0.87%, 1.06%].
Difference-in-Differences (DiD) result: Analyzing the historical region-level initiative (Jaipur vs. Ajmer) verified the parallel-trends assumption ( p=0.48 ). The historical treatment effect was estimated at +1.51 percentage points (95% CI: [1.44%, 1.59%],  p≈0.0 ).
Segment analysis: Positive treatment effects were consistently observed across all production segments:
Production Lines: Line A (+0.96 pp), Line B (+0.98 pp), Line C (+1.03 pp).
Suppliers: Fresh Suppliers (+1.08 pp), Quality Raw Materials (+1.03 pp), ABC Foods (+0.88 pp).
Shifts: Evening (+1.09 pp), Night (+1.03 pp), Morning (+0.79 pp).
Regression adjustment result: Controlling for production lines, suppliers, shifts, regions, machines, and clean raw material weights reduced experimental variance by 16.76%. The covariate-adjusted treatment effect is +0.99 percentage points (95% CI: [0.90%, 1.07%],  p≈0.0 ), reducing the required sample size from 472 to 393 batches.
3. Risks and Limitations
Sample Ratio Mismatch (SRM) Warning: The significant  p -value ( 0.004 ) indicates a potential assignment bias. The treatment effect estimate, though robust, could be slightly skewed if certain types of batches were systemically routed to Control or Treatment.
Segment-level differences: Multiple comparisons were performed during segment analysis; though all segments showed positive direction, segment-level confidence intervals are wider and should be treated as exploratory.
CUPED Limitation: Regression adjustment is utilized here rather than true CUPED, as baseline historical batch-level outcomes matching individual experiment batches were not available.
4. Final Recommendation
EXTEND THE EXPERIMENT (WITH ENGINEERING INTERVENTION)

Reason: Although the statistical treatment effect of +0.97 pp is highly significant and extremely close to our 1% MDE goal (with the adjusted regression model confirming +0.99 pp), we have detected a statistically significant Sample Ratio Mismatch (SRM) ( p=0.004 ). Proceeding with a full "GO" recommendation is risky until the underlying randomisation delivery bug is identified and resolved. We recommend extending the experiment to gather a balanced, bias-free sample once the assignment pipeline has been fixed by the engineering team.

5. Next Steps
Debug the assignment engine: Investigate why more batches were allocated to the Control group (16,803) than the Treatment group (16,283).
Implement pipeline fixes: Re-run the experiment for an additional 14 days under a corrected, verified 50/50 split ratio.
- Continuous Monitoring: Continue tracking process yield, wastage, and quality rejection across all lines (especially Line C, which shows the highest point estimate of +1.03 pp) to ensure no long-term operational degradation occurs.